# 02 — Phase 1 Clustering on Pre-Computed FMA Features

Run notebook 01 first to generate `models/fma_small_features.pkl`.

**Goal**: Cluster 8k FMA tracks using UMAP + HDBSCAN on hand-crafted audio features.
Genre labels are used only as a *sanity check* — not as training signal.

In [ ]:
import sys
sys.path.insert(0, '..')

import pickle
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from collections import Counter

from anther_ml.cluster import fit_clusters, assign_cluster, save_pipeline, cluster_summary
from anther_ml.similarity import SongIndex
from anther_ml.features import extract_librosa_features

print('imports ok')

## Load saved feature data

In [ ]:
with open('../models/fma_small_features.pkl', 'rb') as f:
    data = pickle.load(f)

X = data['X']                      # (N, 518)
genre_labels = data['genre_labels'] # list of strings
track_ids    = data['track_ids']
track_names  = data['track_names']
artist_names = data['artist_names']

print(f'Feature matrix: {X.shape}')
print(f'Genres: {Counter(genre_labels).most_common(5)}')

## Fit UMAP + HDBSCAN

This takes ~3–5 minutes on CPU (Apple Silicon). UMAP is the bottleneck.

In [ ]:
labels, embedding_2d, scaler, pca, reducer, clusterer = fit_clusters(
    X,
    n_umap_components=32,
    n_neighbors=30,
    min_cluster_size=100,
    cluster_selection_method="eom",
    n_pca_components=100,
)

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
noise_pct   = (labels == -1).mean() * 100
print(f'\nClusters found: {n_clusters}')
print(f'Noise points:   {noise_pct:.1f}%')

## Evaluate: Genre purity per cluster

In [ ]:
from sklearn.metrics import silhouette_score

summary = cluster_summary(labels, genre_labels)

# Silhouette on non-noise points
mask = labels != -1
if mask.sum() > 1:
    sil = silhouette_score(X[mask], labels[mask], sample_size=2000)
    print(f'Silhouette score (non-noise): {sil:.4f}  [higher is better, >0.2 is decent]')

print(f'\n{"Cluster":>8} | {"Size":>6} | Top Genre')
print('-' * 40)
for cid, info in sorted(summary.items()):
    tag = '(noise)' if cid == -1 else ''
    print(f'{cid:>8} | {info["size"]:>6} | {info["top_genre"]}  {tag}')

## Visualize: 2D UMAP coloured by genre

In [ ]:
unique_genres = sorted(set(genre_labels))
palette = sns.color_palette('tab20', n_colors=len(unique_genres))
genre_to_color = {g: palette[i] for i, g in enumerate(unique_genres)}
colors = [genre_to_color[g] for g in genre_labels]

fig, ax = plt.subplots(figsize=(14, 10))
scatter = ax.scatter(
    embedding_2d[:, 0], embedding_2d[:, 1],
    c=colors, s=4, alpha=0.6, linewidths=0
)
# Legend
handles = [
    plt.Line2D([0], [0], marker='o', color='w',
               markerfacecolor=genre_to_color[g], markersize=8, label=g)
    for g in unique_genres
]
ax.legend(handles=handles, title='Genre', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
ax.set_title('UMAP of FMA small — coloured by genre (Phase 1)')
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
plt.tight_layout()
plt.savefig('../models/umap_phase1.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to models/umap_phase1.png')

## Visualize: 2D UMAP coloured by HDBSCAN cluster

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))

noise_mask = labels == -1
ax.scatter(
    embedding_2d[noise_mask, 0], embedding_2d[noise_mask, 1],
    c='lightgrey', s=3, alpha=0.3, label='noise', linewidths=0
)
cluster_ids = [l for l in set(labels) if l != -1]
palette2 = sns.color_palette('husl', n_colors=len(cluster_ids))
for cid, color in zip(cluster_ids, palette2):
    mask = labels == cid
    ax.scatter(
        embedding_2d[mask, 0], embedding_2d[mask, 1],
        c=[color], s=4, alpha=0.7, label=f'C{cid}', linewidths=0
    )

ax.set_title(f'HDBSCAN clusters ({n_clusters} found)')
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
plt.tight_layout()
plt.show()

## Save the pipeline and similarity index

In [ ]:
# Save clustering pipeline
save_pipeline('../models/pipeline_phase1.pkl', scaler, reducer, clusterer, pca=pca)

# Build and save similarity index (raw scaled features)
X_scaled = scaler.transform(X)
metadata = [
    {'track_id': tid, 'name': name, 'artist': artist, 'genre': genre, 'cluster': int(cid)}
    for tid, name, artist, genre, cid
    in zip(track_ids, track_names, artist_names, genre_labels, labels)
]
index = SongIndex(X_scaled, metadata)
index.save('../models/index_phase1')

# Also save 2d embedding and labels for later notebooks
np.save('../models/embedding_2d_phase1.npy', embedding_2d)
np.save('../models/labels_phase1.npy', labels)

print('Saved pipeline, index, and 2D embedding.')

## Test: assign a personal MP3 to a cluster

Put the path to any MP3 below. The cell will extract FMA-compatible features,
project into UMAP space, and find the nearest cluster + top-10 similar tracks.

In [ ]:
# ── CHANGE THIS ──────────────────────────────────────────────────────────────
MY_SONG = '/Users/mwilliams/Downloads/icy_.mp3'
# ─────────────────────────────────────────────────────────────────────────────

from anther_ml.cluster import load_pipeline, assign_cluster
from anther_ml.similarity import SongIndex

scaler, pca, reducer, clusterer = load_pipeline('../models/pipeline_phase1.pkl')
index = SongIndex.load('../models/index_phase1')

print('Extracting features...')
vec = extract_librosa_features(MY_SONG)

cluster_id, strength = assign_cluster(vec, scaler, reducer, clusterer, pca=pca)
print(f'\nCluster assignment: {cluster_id}  (strength: {strength:.3f})')
try:
    print(f'Dominant genre in this cluster: {summary[cluster_id]["top_genre"]}')
except NameError:
    pass  # summary not in scope if cells 5-7 were skipped

print('\nTop 10 nearest FMA tracks:')
results = index.query(scaler.transform(vec.reshape(1, -1))[0], top_k=10)
for r in results:
    print(f"  [{r['rank']:2}] score={r['score']:.4f}  {r['artist']} — {r['name']}  ({r['genre']})")

In [ ]:
# Plot the new song as a red dot on the UMAP
vec_scaled = scaler.transform(vec.reshape(1, -1))
song_2d = reducer.transform(vec_scaled)

fig, ax = plt.subplots(figsize=(12, 8))
ax.scatter(embedding_2d[:, 0], embedding_2d[:, 1], c=colors, s=3, alpha=0.4, linewidths=0)
ax.scatter(song_2d[0, 0], song_2d[0, 1], c='red', s=120, zorder=5, label='Your song', edgecolors='black')
ax.legend()
ax.set_title('Your song on the FMA feature map (Phase 1)')
plt.tight_layout()
plt.show()